In [1]:
import numpy as np
import json
import pandas as pd
# from tensorflow.keras.metrics import MeanSquaredError
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv3D
from sklearn.model_selection import KFold 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from scipy.stats import kendalltau
import seaborn as sns
from tqdm import tqdm
import os

In [2]:
lookback = 1
forecast_horizon = 1

In [ ]:

#Where the entire dataset split into its timestamps is stored
timestamps_directory = 'split_files_cleanedVEG/'

#Where the list of all timestamps are stored
timestamps_file_path = os.path.join(timestamps_directory, 'alltimestamps_cleanedVEG.json')

#Where the individual samples are stored
saved_files = 'lookback_and_lookahead_files_cleanedVEG/'

#Where the list of timestamps in their splits are stored
split_file = 'timestamps_splits_cleanedVEG.npz'

In [5]:
# Utility to load all timestamps
def load_all_timestamps():
    with open(timestamps_file_path, 'r') as file:
        timestamps = json.load(file)
        sorted_timestamps = sorted(timestamps)
        return sorted_timestamps

# Utilities to find lengths
def load_split_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    return train_len, train_len + val_len  # Train length and cumulative Train+Val length

def get_all_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    test_len = len(loaded_data['test'])
    return train_len, val_len, test_len

# Function for loading a single training sample
def load_singular_train_data(index, lookback):
    all_timestamps = load_all_timestamps()
    timestamp_name = all_timestamps[lookback + index]
    file_name = f'{index + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single validation sample
def load_singular_val_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, _ = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_len + index]
    file_name = f'{index + train_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single test sample
def load_singular_test_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, train_val_len = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_val_len + index]
    file_name = f'{index + train_val_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    try:
        data = np.load(file_path, allow_pickle=True)
    except:
        print("The last erroneous files don't exist")

    return data['X_batches'], data['y_batches']

In [ ]:
def fill_none_with_mean(values):
    array = np.array([np.nan if v is None else v for v in values])
    nan_indices = np.isnan(array)
    non_nan_indices = np.where(~nan_indices)[0]
    non_nan_values = array[non_nan_indices]
    if len(non_nan_values) == 0: 
        return np.zeros_like(array).tolist()
    array[nan_indices] = np.interp(np.where(nan_indices)[0], non_nan_indices, non_nan_values)
    return array.tolist()

In [ ]:
#Implementation of the TimeSeriesDataset

from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, indices, lookback, mode="train"):
        """
        The parameters are: 
        indices is a list of indices, 
        lookback is set manually, 
        mode to indicate how we are appropriately adding the indices.
        
        """
        self.indices = indices
        self.lookback = lookback
        self.mode = mode
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        index = self.indices[idx]

        # Load data based on mode
        if self.mode == "train":
            X, y = load_singular_train_data(index, self.lookback)
        elif self.mode == "val":
            X, y = load_singular_val_data(index, self.lookback)
        elif self.mode == "test":
            X, y = load_singular_test_data(index, self.lookback)

        for i, array in enumerate(X):
            for j, sub_array in enumerate(array):
                X[i][j] = fill_none_with_mean(sub_array)
        X = np.array(X)  

        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        
        return X_tensor, y_tensor


In [8]:
#Getting lengths of the training, validation and testing
train_length, val_length, test_length = get_all_lengths()

In [ ]:
# Split data indices for train, validation, and test
train_length, val_length, test_length = get_all_lengths()

train_indices = list(range(0, train_length))
val_indices = list(range(0, val_length))
test_indices = list(range(0, test_length))

# Initialize Datasets
train_dataset = TimeSeriesDataset(train_indices, lookback, mode="train")
val_dataset = TimeSeriesDataset(val_indices, lookback, mode="val")
test_dataset = TimeSeriesDataset(test_indices, lookback, mode="test")

# Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [11]:
#Variable names
variable_names = ['10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature', '2 metre temperature', 'UV visible albedo for direct radiation (climatological)', 'Total column rain water', 'Volumetric soil water layer 1', 'Leaf area index, high vegetation', 'Leaf area index, low vegetation', 'Forecast surface roughness', 'Total precipitation', 'Time-integrated surface latent heat net flux', 'Evaporation']

In [ ]:
#MLP definition
class MLP_5D(nn.Module):
    def __init__(self, height, width):
        super(MLP_5D, self).__init__()
        # Define the fully connected layers
        self.fc1 = nn.Linear(8, 128)  # Input channels = 41, output features = 128
        self.dropout1 = nn.Dropout(0.05)
        self.fc2 = nn.Linear(128, 64)  # Output features = 64
        self.dropout2 = nn.Dropout(0.05)
        self.fc3 = nn.Linear(64, 1)    # Final output, reducing to 1 channel

        self.height = height
        self.width = width

    def forward(self, x):
        batch_size, timesteps, channels, height, width = x.shape
        
        # Ensure the input spatial dimensions match the expected height and width
        assert height == self.height and width == self.width, "Height and width mismatch"
        
        # Reshape to (batch * timesteps * height * width, channels)
        x = x.permute(0, 1, 3, 4, 2).reshape(-1, channels)
        
        # Apply MLP
        x = self.fc1(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = torch.nn.functional.softplus(x)
        
        # Reshape back to (batch, timesteps, 1, height, width)
        x = x.view(batch_size, timesteps, self.height, self.width, 1).permute(0, 1, 4, 2, 3)

        return x

In [ ]:
#ConvLSTM definition
from ConvLSTM_PERIODIC import ConvLSTM_PERIODIC
import torch
import torch.nn as nn
from collections import defaultdict

class ConvLSTMNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims, kernel_size, num_layers, output_channels, batch_first=True, pool_size=(2,2)):
        super(ConvLSTMNetwork, self).__init__()
        
        # ConvLSTM module
        self.convlstm = ConvLSTM_PERIODIC(input_dim=input_dim,
                                 hidden_dim=hidden_dims,
                                 kernel_size=kernel_size,
                                 num_layers=num_layers,
                                 batch_first=batch_first,
                                 bias=True,
                                 return_all_layers=True)
        
        # Batch Normalization for each ConvLSTM layer's output
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm3d(hidden_dim) for hidden_dim in hidden_dims
        ])

        # Final Conv3D layer for regression pathway
        self.conv3d = nn.Conv3d(in_channels=hidden_dims[-1],
                                out_channels=output_channels,
                                kernel_size=(1, 3, 3),
                                padding=(0, 1, 1))

        # MLP for regression output: (B,T,C,H,W) -> (B,T,1,H,W)
        self.mlp = MLP_5D(height=81, width=97)

        # Classification head for pixel-level zero precipitation probability
        # We'll produce (B,T,1,H,W) as well:
        self.classification_head = nn.Sequential(
            nn.Conv3d(output_channels, 1, kernel_size=(1,1,1)),  # from C to 1 channel
            nn.Sigmoid()
        )

        self.activation_variance = defaultdict(list)

    def forward(self, x):
        """
        x: (B, T, input_dim, H, W)
        """
        # Forward through ConvLSTM
        layer_output_list, last_state_list = self.convlstm(x)
        
        # Apply batch norms
        for i, output in enumerate(layer_output_list):
            # output: (B, T, C, H, W)
            output = output.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W) for BatchNorm3d
            output = self.batch_norms[i](output)
            output = output.permute(0, 2, 1, 3, 4)  # back to (B, T, C, H, W)

            #Track variance across spatial dimensions for hooks with activation tracking 
            activation_variance = output.var(dim=(3, 4)).mean().item()
            self.activation_variance[f"ConvLSTM_layer_{i}"].append(activation_variance)

            layer_output_list[i] = output
        
        # Take output from the last ConvLSTM layer
        final_output = layer_output_list[-1]  # (B, T, C, H, W)

        # Pass through Conv3D: needs (B,C,T,H,W)
        final_output = final_output.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
        final_output = self.conv3d(final_output)

        # Return to (B,T,C,H,W) for MLP (regression)
        final_output_t = final_output.permute(0, 2, 1, 3, 4)  # (B,T,C,H,W)

        # Regression output
        regression_output = self.mlp(final_output_t)  # (B,T,1,H,W)

        # Classification output:
        # The classification head is defined for (B,C,T,H,W), so reorder again
        final_output_c = final_output  # still (B,output_channels,T,H,W)
        classification_output = self.classification_head(final_output_c)
        # classification_output: (B,1,T,H,W)

        # Permute classification output to match (B,T,1,H,W) format
        classification_output = classification_output.permute(0, 2, 1, 3, 4)  # (B,T,1,H,W)

        return regression_output, classification_output

In [14]:
# Setting device
if torch.cuda.is_available():
    print("running on cuda")
    device = torch.device('cuda')
else:
    print("running on the cpu")
    device = torch.device('cpu')

In [ ]:
input_dim = 1
set_lookback = 1
set_forecast_horizon = 1

model = ConvLSTMNetwork(
    input_dim=8 * set_lookback, 
    hidden_dims=[8, 32, 64], 
    kernel_size=(3,3), 
    num_layers=3, 
    output_channels=8 * set_forecast_horizon, 
    batch_first=True
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.005)

loss_fn = nn.MSELoss() 
bce_loss_fn = nn.BCELoss()  

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

train_losses = []
val_losses = []

y_true = []
y_pred = []

train_losses_per_fold = []
val_losses_per_fold = []

height = 81
width = 97

num_epochs = 400
scaling_factor = 1


In [16]:

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:

#Only this variable if use if you want to run the model on certain variables 
exclude_variables = [
    'UV visible albedo for direct radiation (climatological)',
    'Volumetric soil water layer 1',
    'Leaf area index, high vegetation',
    'Leaf area index, low vegetation',
    'Forecast surface roughness',
    'Evaporation']

# Definition of log epsilon
epsilon = 1e-3

# Function to normalize using z-score
def z_score_normalisation(batch, mean, std, epsilon=1e-6):
    return (batch - mean) / (std + epsilon)

# Function to normalize using min-max scaling
def min_max_scaling(batch, min_value, max_value, epsilon=1e-6):
    return (batch - min_value) / (max_value - min_value + epsilon)

# Function to apply log transformation
def log_normalisation(batch, epsilon=1e-3):
    return torch.log(batch + epsilon)

def preprocess_data(data_loader, variable_names, global_stats, name):
    """
    Preprocess the entire dataset with different normalization strategies for each variable.
    """
    log_variables = ['Total precipitation', 'Volumetric soil water layer 1', 'Total column rain water']
    log_stats = {var: {'logs': []} for var in log_variables}  # Dictionary to store logs for each variable

    # ---- First pass Compute log mean and std for specified variables ----
    count_count = 0

    with torch.no_grad():
        for X_batch, _ in tqdm(data_loader, desc=f"Computing log stats for variables"):
            if count_count == len(data_loader) - 2:
                break

            count_count += 1
            X_batch = X_batch.clone()

            for var in log_variables:
                var_idx = variable_names.index(var)
                X_var = X_batch[:, :, var_idx]  # Extract values for the variable
                if var == 'Total precipitation':  # Convert precipitation to mm
                    X_var = X_var * 1000

                log_X_var = torch.log(X_var + epsilon)
                log_stats[var]['logs'].append(log_X_var.flatten().cpu().numpy())
                num_nan = torch.isnan(log_X_var).sum().item()  # Count number of NaN values
                num_inf = torch.isinf(log_X_var).sum().item()  # Count number of Inf values
                if num_nan > 0 or num_inf > 0:
                    print(num_nan, num_inf)

    # Calculate mean and std for log-transformed variables (if needed later)
    for var in log_variables:
        logs_all = np.concatenate(log_stats[var]['logs']) if log_stats[var]['logs'] else np.array([0])
        mean_log = np.mean(logs_all)
        std_log = np.std(logs_all)
        global_stats.setdefault(var, {})  # Ensure the variable exists in global_stats
        global_stats[var]['mean_log'] = mean_log
        global_stats[var]['std_log'] = std_log

    stop_count = 0
    # ---- Second pass: Normalize the data ----
    normalized_batches = []
    for X_batch, y_batch in tqdm(data_loader, desc=f"Preprocessing {name.capitalize()} Data"):
        X_batch = X_batch.clone()
        y_batch = y_batch.clone()

        if stop_count == len(data_loader) - 2:
            break

        stop_count += 1

        new_X_batch = []  # Collect only normalized variables not in exclude_variables

        # Normalize each variable
        for var_idx, variable_name in enumerate(variable_names):
            if variable_name in exclude_variables:
                continue  # Skip excluded variables

            X_var = X_batch[:, :, var_idx]  # Extract the variable

            if variable_name == 'Total precipitation':
                X_var = X_var * 1000  # Convert precipitation to mm
                mean_log = global_stats["Total precipitation"]['mean_log']
                std_log = global_stats["Total precipitation"]['std_log']
                X_var_normalized = z_score_normalisation(
                    log_normalisation(X_var, epsilon),
                    mean_log,
                    std_log
                )

            elif variable_name in ['Volumetric soil water layer 1', 'Total column rain water']:
                mean_log = global_stats[variable_name]['mean_log']
                std_log = global_stats[variable_name]['std_log']
                X_var_normalized = z_score_normalisation(
                    log_normalisation(X_var, epsilon),
                    mean_log,
                    std_log
                )

            elif variable_name == 'Leaf area index, high vegetation' or variable_name == 'Leaf area index, low vegetation':
                mean = global_stats[variable_name]['mean']
                std = global_stats[variable_name]['std']
                X_var_normalized = z_score_normalisation(X_var, mean, std)

            elif variable_name == 'Forecast surface roughness':
                min_value = global_stats[variable_name]['min']
                max_value = global_stats[variable_name]['max']
                X_var_normalized = min_max_scaling(X_var, min_value, max_value)

            elif variable_name == 'Evaporation':
                mean = global_stats[variable_name]['mean']
                std = global_stats[variable_name]['std']
                X_var_normalized = z_score_normalisation(X_var * 1000, mean, std)

            elif variable_name == 'Time-integrated surface latent heat net flux':
                mean = global_stats[variable_name]['mean']
                std = global_stats[variable_name]['std']
                X_var_normalized = z_score_normalisation(X_var / 10800, mean, std)

            elif variable_name in ['2 metre temperature', '2 metre dewpoint temperature', 
                                   '10 metre U wind component', '10 metre V wind component']:
                mean = global_stats[variable_name]['mean']
                std = global_stats[variable_name]['std']
                X_var_normalized = z_score_normalisation(X_var, mean, std)

            # Append the normalized variable to new_X_batch
            new_X_batch.append(X_var_normalized.unsqueeze(2))  # Keep the correct dimension

        # Concatenate all normalized variables along the feature dimension
        final_X_batch = torch.cat(new_X_batch, dim=2)

        # Add zero indicator to the final input features
        precip_idx = variable_names.index('Total precipitation')
        zero_indicator = (X_batch[:, :, precip_idx] == 0).float()
        final_X_batch = torch.cat((final_X_batch, zero_indicator.unsqueeze(2)), dim=2)

        y_zero_indicator = (y_batch == 0).float()
        y_batch = y_batch * 1000  # Convert target precipitation to mm

        # Append to normalized_batches
        normalized_batches.append((final_X_batch, y_batch, y_zero_indicator))

    return normalized_batches

with open('globaldatastatistics.json', 'r') as f:
    global_stats = json.load(f)

# Precompute normalized data
normalized_train_data = preprocess_data(train_loader, variable_names, global_stats, "train")
normalized_val_data = preprocess_data(val_loader, variable_names, global_stats, "val")
normalized_test_data = preprocess_data(test_loader, variable_names, global_stats, "test")




In [ ]:
# Save the normalized data
torch.save(normalized_train_data, "normalized_train_data.pth")
torch.save(normalized_val_data, "normalized_val_data.pth")
torch.save(normalized_test_data, "normalized_test_data.pth")

In [21]:
#Definition of evaluation metrics
from scipy.stats import pearsonr, spearmanr
def nash_sutcliffe_efficiency(observed, predicted):
    # Ensure inputs are tensors on the CPU
    observed = observed.cpu()
    predicted = predicted.cpu()

    # Compute the numerator and denominator
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)

    # Calculate NSE
    nse = 1 - (numerator / denominator)
    return nse.item()

from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return pearsonr(y_true, y_pred)[0]  # Return the correlation coefficient

from scipy.stats import spearmanr

def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return spearmanr(y_true, y_pred).correlation  # Return the Spearman correlation

def mse(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean((y_true - y_pred) ** 2).item()

def mae(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean(torch.abs(y_true - y_pred)).item()

def percentage_error(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

def percentage_bias(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

import torch.nn.functional as F

def earth_movers_distance(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    # Compute EMD using the Wasserstein distance (L1 distance)
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return kendalltau(y_true, y_pred).correlation  # Return the Kendall Tau

def r2_score(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
def plot_activation_variances(activation_variances):
    plt.figure(figsize=(10, 6))
    for layer, values in activation_variances.items():
        plt.plot(values, label=layer)
    plt.xlabel("Training Step")
    plt.ylabel("Activation Variance")
    plt.title("Activation Variance Across Layers")
    plt.legend()
    plt.show()


In [ ]:
# Load the normalized data
normalized_train_data = torch.load("normalized_train_data.pth")
normalized_val_data = torch.load("normalized_val_data.pth")
normalized_test_data = torch.load("normalized_test_data.pth")

print("Loaded normalized data successfully!")


In [ ]:
best_val_loss = float('inf')  # Initialize with a very large value
best_y_pred = None
best_y_true = None
best_y_pred_corr = None
best_y_true_corr = None
best_spatial_corr = 0
best_classification_accuracy = 0  # Initialize for classification accuracy

# Weighting factor for classification loss
alpha = 1.0  # Adjust as needed for balancing regression and classification

for epoch in range(100):
    model.train()
    running_loss = 0.0
    running_reg_loss = 0.0
    running_class_loss = 0.0
    running_class_accuracy = 0.0  # Track classification accuracy
    running_spatial_corr = 0.0  # To accumulate spatial correlations
    spatial_corr_count = 0  # To count the number of spatial correlation values

    for X_batch, y_batch, y_zero_indicator in tqdm(normalized_train_data, desc=f'Epoch {epoch+1}/{num_epochs} Training'):

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_zero_indicator = y_zero_indicator.to(device)

        # Reshape for model input
        batch_size, time_steps_in, channels_in, grid_points = X_batch.shape
        batch_size, time_steps_out, channels_out, grid_points = y_batch.shape
        X_batch = X_batch.view(batch_size, time_steps_in, channels_in, height, width)
        y_batch = y_batch.view(batch_size, time_steps_out, channels_out, height, width)
        y_zero_indicator = y_zero_indicator.view(batch_size, time_steps_out, channels_out, height, width)

        optimizer.zero_grad()

        regression_output, classification_output = model(X_batch)

        reg_loss = loss_fn(regression_output, y_batch)
        class_loss = bce_loss_fn(classification_output, y_zero_indicator)
        total_loss = reg_loss + alpha * class_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item() * X_batch.size(0)
        running_reg_loss += reg_loss.item() * X_batch.size(0)
        running_class_loss += class_loss.item() * X_batch.size(0)

        # Compute classification accuracy
        classification_predictions = (classification_output 
                                      ).float()  # Threshold for binary classification
        correct_classifications = (classification_predictions == y_zero_indicator).sum().item()
        total_classifications = y_zero_indicator.numel()
        running_class_accuracy += correct_classifications / total_classifications

        # Compute spatial correlation for regression
        batch_spatial_corr = spatial_correlation(y_batch, regression_output)
        running_spatial_corr += batch_spatial_corr
        spatial_corr_count += 1

    # Compute averages for training metrics
    train_loss = running_loss / len(normalized_train_data)
    train_reg_loss = running_reg_loss / len(normalized_train_data)
    train_class_loss = running_class_loss / len(normalized_train_data)
    train_spatial_corr = running_spatial_corr / spatial_corr_count
    train_class_accuracy = running_class_accuracy / len(normalized_train_data)

    train_losses.append(train_loss)

    print(f"Epoch {epoch + 1}/{num_epochs} Training Loss: {train_loss:.6f}")
    print(f"Training Regression Loss: {train_reg_loss:.6f}")
    print(f"Training Classification Loss: {train_class_loss:.6f}")
    print(f"Training Classification Accuracy: {train_class_accuracy:.6f}")
    print(f"Epoch {epoch + 1}/{num_epochs} Spatial Correlation: {train_spatial_corr:.6f}")

    # Validation loop
    model.eval()
    val_loss = 0.0
    val_reg_loss = 0.0
    val_class_loss = 0.0
    running_val_spatial_corr = 0.0
    running_val_class_accuracy = 0.0
    val_corr_count = 0

    with torch.no_grad():
        for X_val, y_val, y_zero_val in tqdm(normalized_val_data, desc=f'Epoch {epoch+1}/{num_epochs} Validation'):
            X_val = X_val.to(device)
            y_val = y_val.to(device)
            y_zero_val = y_zero_val.to(device)

            batch_size, time_steps_in, channels_in, grid_points = X_val.shape
            batch_size, time_steps_out, channels_out, grid_points = y_val.shape
            X_val = X_val.view(batch_size, time_steps_in, channels_in, height, width)
            y_val = y_val.view(batch_size, time_steps_out, channels_out, height, width)
            y_zero_val = y_zero_val.view(batch_size, time_steps_out, channels_out, height, width)

            regression_output, classification_output = model(X_val)

            # Compute validation losses
            reg_loss = loss_fn(regression_output, y_val)
            class_loss = bce_loss_fn(classification_output, y_zero_val)
            total_loss = reg_loss + alpha * class_loss

            val_loss += total_loss.item() * X_val.size(0)
            val_reg_loss += reg_loss.item() * X_val.size(0)
            val_class_loss += class_loss.item() * X_val.size(0)

            # Compute classification accuracy
            classification_predictions = (classification_output > 0.5).float()
            correct_classifications = (classification_predictions == y_zero_val).sum().item()
            total_classifications = y_zero_val.numel()
            running_val_class_accuracy += correct_classifications / total_classifications

            spatial_corr = spatial_correlation(y_val, regression_output)
            running_val_spatial_corr += spatial_corr
            val_corr_count += 1

    # Compute averages for validation metrics
    val_loss /= len(normalized_val_data)
    val_reg_loss /= len(normalized_val_data)
    val_class_loss /= len(normalized_val_data)
    avg_val_spatial_corr = running_val_spatial_corr / val_corr_count
    val_class_accuracy = running_val_class_accuracy / len(normalized_val_data)
    val_losses.append(val_loss)

    print(f"Epoch {epoch + 1}/{num_epochs} Validation Loss: {val_loss:.6f}")
    print(f"Validation Regression Loss: {val_reg_loss:.6f}")
    print(f"Validation Classification Loss: {val_class_loss:.6f}")
    print(f"Validation Classification Accuracy: {val_class_accuracy:.6f}")
    print(f"Epoch {epoch + 1}/{num_epochs} Validation Spatial Correlation: {avg_val_spatial_corr:.6f}")

    # Save best validation results
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_y_pred = regression_output
        best_y_true = y_val

    if spatial_corr > best_spatial_corr:
        best_spatial_corr = spatial_corr
        best_y_pred_corr = regression_output
        best_y_true_corr = y_val

    if val_class_accuracy > best_classification_accuracy:
        best_classification_accuracy = val_class_accuracy
        best_y_pred_class = classification_output
        best_y_true_class = y_zero_val

    scheduler.step(val_loss)

# Save the best validation predictions and true values
torch.save({
    'y_true_loss': best_y_true,
    'y_pred_loss': best_y_pred,
    'y_true_corr': best_y_true_corr,
    'y_pred_corr': best_y_pred_corr,
    'y_true_class': best_y_true_class,
    'y_pred_class': best_y_pred_class,
}, 'best_validation_resultsMULTI_TASK.pth')

print("Best validation predictions and ground truth saved.")

# Save the model state dictionary
model_path = "NEWPIPELINEConvLSTM_MULTI_TASK"
torch.save({
    'model_state_dict': model.state_dict(),
}, model_path)



In [ ]:
# Save the best validation predictions and true values
torch.save({
    'y_true_loss': best_y_true,
    'y_pred_loss': best_y_pred,
    'y_true_corr': best_y_true_corr,
    'y_pred_corr': best_y_pred_corr,
    'y_true_class': best_y_true_class,
    'y_pred_class': best_y_pred_class,
}, 'best_validation_resultsMULTI_TASK.pth')

print("Best validation predictions and ground truth saved.")

In [ ]:
# Save the model state dictionary
model_path = "NEWPIPELINEConvLSTM_MULTI_TASK"
torch.save({
    'model_state_dict': model.state_dict(),
}, model_path)

In [ ]:
#Visualise the training and validation losses

import matplotlib.pyplot as plt

# Plot the training and validation loss for all of the folds combined
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epochs")
plt.ylabel("Training Dice Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.title("Validation Loss")
plt.legend()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm  # Progress bar

# Saliency maps 
def compute_average_saliency_map(model, validation_data):
    """
    Computes the average saliency map across the entire validation dataset.
    """
    total_saliency = np.zeros((channels_in, height, width))
    total_samples = 0

    print("\nComputing average saliency map...")
    for batch_idx, (X_val, y_val, _) in enumerate(tqdm(validation_data, desc="Batches")):
        X_val = X_val.to(device)
        X_val = X_val.view(batch_size, time_steps_in, channels_in, height, width)

        for X_input in X_val:
            # Ensure proper shape and gradient computation
            X_input = X_input.unsqueeze(0).requires_grad_().to(device)
            model.zero_grad()

            # Forward pass
            regression_output, _ = model(X_input)
            loss = regression_output.sum()  # Sum all outputs to get gradients
            loss.backward()

            # Compute absolute gradients (saliency map)
            saliency = X_input.grad.abs().cpu().numpy()
            total_saliency += saliency[0, 0]  # Accumulate saliency maps for this sample
            total_samples += 1

    average_saliency = total_saliency / total_samples

    # Plot saliency maps for all features
    for feature_idx in range(channels_in-1):
        plt.figure(figsize=(10, 6))
        plt.imshow(average_saliency[feature_idx], cmap='hot', interpolation='nearest')
        plt.title(f"Averaged Saliency Map for Feature {variable_names[feature_idx]} (Validation Set)")
        plt.colorbar()
        plt.show()

    return average_saliency

def compute_average_weights(model):
    """
    Computes the average weight distributions across the entire model.
    """
    weights_dict = {}  # To store weights of all layers

    print("\nComputing average weights for all layers...")
    for name, param in tqdm(model.named_parameters(), desc="Layers"):
        if "weight" in name and param.requires_grad:
            weights = param.data.cpu().numpy().flatten()
            if name not in weights_dict:
                weights_dict[name] = []
            weights_dict[name].extend(weights)

    # Plot average weight distributions for each layer
    for layer, weights in weights_dict.items():
        plt.figure(figsize=(10, 6))
        plt.hist(weights, bins=50, color='blue', alpha=0.7)
        plt.title(f"Average Weight Distribution for Layer: {layer}")
        plt.xlabel("Weight Value")
        plt.ylabel("Frequency")
        plt.show()


def evaluate_validation_batches(model, validation_data):
    """
    Evaluates the model by computing average saliency maps and average weight distributions.
    """
    print(f"\nEvaluating validation data with progress tracking...")
    compute_average_saliency_map(model, validation_data)

    compute_average_weights(model)

    print("Completed evaluation for all validation batches.")

evaluate_validation_batches(model, normalized_val_data)


In [31]:
#Definition of evaluation metrics
from scipy.stats import pearsonr, spearmanr
def nash_sutcliffe_efficiency(observed, predicted):
    # Ensure inputs are tensors on the CPU
    observed = observed.cpu()
    predicted = predicted.cpu()

    # Compute the numerator and denominator
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)

    # Calculate NSE
    nse = 1 - (numerator / denominator)
    return nse.item()

from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return pearsonr(y_true, y_pred)[0]  # Return the correlation coefficient

from scipy.stats import spearmanr

def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return spearmanr(y_true, y_pred).correlation  # Return the Spearman correlation

def mse(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean((y_true - y_pred) ** 2).item()

def mae(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean(torch.abs(y_true - y_pred)).item()

def percentage_error(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

def percentage_bias(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

import torch.nn.functional as F

def earth_movers_distance(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    # Compute EMD using the Wasserstein distance (L1 distance)
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return kendalltau(y_true, y_pred).correlation  # Return the Kendall Tau

def r2_score(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:

input_dim = 8
set_lookback = 1
set_forecast_horizon = 1

model = ConvLSTMNetwork(
    input_dim=8 * set_lookback, 
    hidden_dims=[8, 32, 64], 
    kernel_size=(3,3), 
    num_layers=3, 
    output_channels=8 * set_forecast_horizon, 
    batch_first=True
)

model = model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

loss_fn = nn.MSELoss()    
bce_loss_fn = nn.BCELoss()    

optimizer = optim.AdamW(model.parameters(), lr = 0.005) 
checkpoint = torch.load('NEWPIPELINEConvLSTM_MULTI_TASK_NOVEG')
model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)  
model.eval()

print("Model loaded successfully")


In [33]:

from scipy.stats import pearsonr, spearmanr
batch_size = batch_size
time_steps_out = set_forecast_horizon
channels = 8
height = 81
width = 97
scaling_factor = 1

In [ ]:
def denormalize_precipitation(normalized_values, mean_log, std_log):
    """
    Denormalize precipitation data that was log-normalized and standardized.

    Parameters:
        normalized_values (torch.Tensor or np.ndarray): Normalized precipitation values.
        mean_log (float): Mean of log-transformed precipitation.
        std_log (float): Standard deviation of log-transformed precipitation.

    Returns:
        denormalized_values (np.ndarray): Denormalized precipitation values in mm.
    """
    log_values = (normalized_values * std_log) + mean_log

    denormalized_values = np.exp(log_values) - epsilon

    return denormalized_values


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

best_validation_predictions_and_ground_truth = torch.load('best_validation_resultsMULTI_TASK_NOTVEGREAL.pth')

y_true = best_validation_predictions_and_ground_truth['y_true_loss']
y_pred = best_validation_predictions_and_ground_truth['y_pred_loss']

y_true_class = best_validation_predictions_and_ground_truth['y_true_class']
y_pred_class = best_validation_predictions_and_ground_truth['y_pred_class']
y_predicted = y_pred
y_actual = y_true

y_pred_class_array = y_pred_class.cpu().numpy().flatten() 
y_true_class_array = y_true_class.cpu().numpy().flatten()  

# Performance metrics for regression
metrics = {
    "NSE": nash_sutcliffe_efficiency(y_actual, y_predicted),
    "R2": r2_score(y_actual, y_predicted),
    "Pearson": pearson_correlation(y_actual, y_predicted),
    "Spearman": spearman_correlation(y_actual, y_predicted),
    "MSE": mse(y_actual, y_predicted),
    "MAE": mae(y_actual, y_predicted),
    "Percentage Error": percentage_error(y_actual, y_predicted),
    "Percentage Bias": percentage_bias(y_actual, y_predicted),
    "EMD": earth_movers_distance(y_actual, y_predicted),
    "Kendall Tau": kendall_tau(y_actual, y_predicted),
    "Spatial Correlation": spatial_correlation(y_actual, y_predicted)
}

# Classification metrics
classification_metrics = {
    "Accuracy": accuracy_score(y_true_class_array, (y_pred_class_array > 0.5)),
    "Precision": precision_score(y_true_class_array, (y_pred_class_array > 0.5)),
    "Recall": recall_score(y_true_class_array, (y_pred_class_array > 0.5)),
    "F1-Score": f1_score(y_true_class_array, (y_pred_class_array > 0.5)),
    # "ROC-AUC": roc_auc_score(y_true_class_array, y_pred_class_array),
}

# Print regression metrics
print("\nRegression Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.16f}")

# Print classification metrics
print("\nClassification Metrics:")
for metric, value in classification_metrics.items():
    print(f"{metric}: {value:.16f}")

# Calculate mean and standard deviation of regression predictions
y_pred_array = np.array(y_pred.cpu())
y_true_array = np.array(y_true.cpu())

y_true_mean = np.mean(y_true_array)
y_pred_mean = np.mean(y_pred_array)

y_true_std = np.std(y_true_array)
y_pred_std = np.std(y_pred_array)

# Print mean and standard deviation
print("\nMean and SD of Ground Truth Precipitation:")
print(f"Mean: {y_true_mean:.16f}, SD: {y_true_std:.16f}")

print("\nMean and SD of Predicted Precipitation:")
print(f"Mean: {y_pred_mean:.16f}, SD: {y_pred_std:.16f}")



In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch 

best_validation_predictions_and_ground_truth = torch.load('best_validation_resultsMULTI_TASK_NOTVEGREAL.pth')

# Regression outputs
y_true = best_validation_predictions_and_ground_truth['y_true_corr']
y_pred = best_validation_predictions_and_ground_truth['y_pred_corr']

# Classification outputs
y_true_class = best_validation_predictions_and_ground_truth['y_true_class']
y_pred_class = best_validation_predictions_and_ground_truth['y_pred_class']

def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()

def plot_classification_rates(y_true_class, y_pred_class):
    """
    Plot the classification metrics: True Positive Rate, False Positive Rate, etc.
    """
    y_pred_binary = (y_pred_class > 0.5).astype(int)

    tp = np.sum((y_pred_binary == 1) & (y_true_class == 1))
    fp = np.sum((y_pred_binary == 1) & (y_true_class == 0))
    tn = np.sum((y_pred_binary == 0) & (y_true_class == 0))
    fn = np.sum((y_pred_binary == 0) & (y_true_class == 1))

    total = tp + fp + tn + fn
    rates = {
        "True Positive Rate": tp / total,
        "False Positive Rate": fp / total,
        "True Negative Rate": tn / total,
        "False Negative Rate": fn / total
    }

    plt.figure(figsize=(8, 6))
    sns.barplot(x=list(rates.keys()), y=list(rates.values()), palette="viridis")
    plt.title("Classification Rates")
    plt.ylabel("Rate")
    plt.xlabel("Metric")
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.show()

def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title):
    """
    Heatmap for spatial precipitation data.
    """
    mean_precip = np.mean(data, axis=0)
    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

# Flatten regression outputs for easier plotting
y_true_flat = y_true.cpu().numpy().flatten()
y_pred_flat = y_pred.cpu().numpy().flatten()

# Flatten classification outputs
y_true_class_flat = y_true_class.cpu().numpy().flatten()
y_pred_class_flat = y_pred_class.cpu().numpy().flatten()

# Hexbin plots for regression
plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_flat, y_pred_flat, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin plot of ground truth vs predicted precipitation")
plt.show()

# Histograms for regression
plot_precipitation_distribution(y_true_flat, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_flat, "Predicted Precipitation")

# Scatter plot for regression
plot_scatter(y_true_flat, y_pred_flat, "Scatter Plot of Ground Truth vs Predicted Precipitation")

# Classification rates plot
plot_classification_rates(y_true_class_flat, y_pred_class_flat)

# Heatmaps for spatial data
plot_spatial_heatmap(y_true[0, 0].cpu().numpy(), "Ground Truth Spatial Precipitation (First Timestamp)")
plot_spatial_heatmap(y_pred[0, 0].cpu().numpy(), "Predicted Spatial Precipitation (First Timestamp)")

# Heatmaps for classification
plot_spatial_heatmap(y_true_class[0, 0].cpu().numpy(), "Ground Truth Zero Classification (First Timestamp)")
plot_spatial_heatmap(y_pred_class[0, 0].cpu().numpy(), "Predicted Zero Classification (First Timestamp)")




In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

threshold = 0.1
precip_index = 10

def evaluate(model, test_loader, reg_loss_fn, class_loss_fn, device, variable_names, height, width, scaling_factor):
    """
    Evaluate the model on the test set for both regression and classification tasks.
    """
    model.eval()  # Set the model to evaluation model

    input_to_true = {'zero_to_non_zero': 0, 'non_zero_to_zero': 0}
    input_to_pred_REG = {'zero_to_non_zero': 0, 'non_zero_to_zero': 0}
    input_to_pred_CLASS = {'zero_to_non_zero': 0, 'non_zero_to_zero': 0}

    test_reg_loss = 0.0
    test_class_loss = 0.0
    test_total_loss = 0.0

    y_true_reg = []
    y_pred_reg = [] 

    y_true_class = [] 
    y_pred_class = []  

    with torch.no_grad():
        for X_test, y_test, y_zero_test in tqdm(test_loader, desc="Evaluating on Test Set"):
            X_test, y_test, y_zero_test = X_test.to(device), y_test.to(device), y_zero_test.to(device)

            batch_size, time_steps_in, channels_in, grid_points = X_test.shape
            batch_size, time_steps_out, channels_out, grid_points = y_test.shape
            X_test = X_test.view(batch_size, time_steps_in, channels_in, height, width)
            y_test = y_test.view(batch_size, time_steps_out, channels_out, height, width)
            y_zero_test = y_zero_test.view(batch_size, time_steps_out, channels_out, height, width)

            regression_output, classification_output = model(X_test)

            # Compute regression loss
            reg_loss = reg_loss_fn(regression_output, y_test)

            # Compute classification loss
            class_loss = class_loss_fn(classification_output, y_zero_test)

            # Total loss
            total_loss = reg_loss + scaling_factor * class_loss

            X_precip = X_test[:, :, 5]
            y_true = y_test
            y_pred = regression_output

            zero_mask_X = X_precip <= threshold  # Input precipitation ≤ 0.1mm
            non_zero_mask_X = X_precip > threshold  # Input precipitation > 0.1mm

            zero_mask_y_true = y_true <= threshold  # True precipitation ≤ 0.1mm
            non_zero_mask_y_true = y_true > threshold  # True precipitation > 0.1mm

            zero_mask_y_pred = regression_output <= threshold  # Predicted precipitation ≤ 0.1mm
            non_zero_mask_y_pred = regression_output > threshold  # Predicted precipitation > 0.1mm

            zero_mask_y_pred_CLASS = classification_output <= 0.5
            non_zero_mask_y_pred_CLASS = classification_output > 0.5

            # Transitions: Input to True
            input_to_true['zero_to_non_zero'] += ((zero_mask_X & non_zero_mask_y_true).sum().item())
            input_to_true['non_zero_to_zero'] += ((non_zero_mask_X & zero_mask_y_true).sum().item())

            # Transitions: Input to Predicted
            input_to_pred_REG['zero_to_non_zero'] += ((zero_mask_X & non_zero_mask_y_pred).sum().item())
            input_to_pred_REG['non_zero_to_zero'] += ((non_zero_mask_X & zero_mask_y_pred).sum().item())

            input_to_pred_CLASS['zero_to_non_zero'] += ((zero_mask_X & non_zero_mask_y_pred_CLASS).sum().item())
            input_to_pred_CLASS['non_zero_to_zero'] += ((non_zero_mask_X & zero_mask_y_pred_CLASS).sum().item())

            # Accumulate losses
            test_reg_loss += reg_loss.item() * X_test.size(0)
            test_class_loss += class_loss.item() * X_test.size(0)
            test_total_loss += total_loss.item() * X_test.size(0)

            # Collect true and predicted values for regression and classification
            y_true_reg.append(y_test.cpu())
            y_pred_reg.append(regression_output.cpu())
            y_true_class.append(y_zero_test.cpu())
            y_pred_class.append(classification_output.cpu())

    # Normalize losses by the total dataset size
    test_reg_loss /= len(test_loader)
    test_class_loss /= len(test_loader)
    test_total_loss /= len(test_loader)

    print(f"Test Regression Loss: {test_reg_loss:.16f}")
    print(f"Test Classification Loss: {test_class_loss:.16f}")
    print(f"Test Total Loss: {test_total_loss:.16f}")

    y_true_reg_flat = torch.cat(y_true_reg, dim=0).flatten()  # Keep as PyTorch tensor
    y_pred_reg_flat = torch.cat(y_pred_reg, dim=0).flatten()  # Keep as PyTorch tensor
    y_true_class_flat = torch.cat(y_true_class, dim=0).flatten()  # Keep as PyTorch tensor
    y_pred_class_flat = torch.cat(y_pred_class, dim=0).flatten()  # Keep as PyTorch tensor

    # Compute regression metrics
    regression_metrics = {
        "MSE": mse(y_true_reg_flat, y_pred_reg_flat),
        "MAE": mae(y_true_reg_flat, y_pred_reg_flat),
        "NSE": nash_sutcliffe_efficiency(y_true_reg_flat, y_pred_reg_flat),
        "R2": r2_score(y_true_reg_flat, y_pred_reg_flat),
        "Pearson": pearson_correlation(y_true_reg_flat, y_pred_reg_flat),
        "Spearman": spearman_correlation(y_true_reg_flat, y_pred_reg_flat),
        "NSE": nash_sutcliffe_efficiency(y_true_reg_flat, y_pred_reg_flat),
        "Percentage Error": percentage_error(y_true_reg_flat, y_pred_reg_flat),
        "Percentage Bias": percentage_bias(y_true_reg_flat, y_pred_reg_flat),
        "EMD": earth_movers_distance(y_true_reg_flat, y_pred_reg_flat),
        "Kendall Tau": kendall_tau(y_true_reg_flat, y_pred_reg_flat),
        "Spatial Correlation": spatial_correlation(y_true_reg_flat, y_pred_reg_flat)}

    print("\nRegression Metrics:")
    for metric, value in regression_metrics.items():
        print(f"{metric}: {value:.16f}")

    # Compute classification metrics
    classification_metrics = {
        "Accuracy": accuracy_score(y_true_class_flat, (y_pred_class_flat > 0.5)),
        "Precision": precision_score(y_true_class_flat, (y_pred_class_flat > 0.5)),
        "Recall": recall_score(y_true_class_flat, (y_pred_class_flat > 0.5)),
        "F1": f1_score(y_true_class_flat, (y_pred_class_flat > 0.5)),
        # "ROC-AUC": roc_auc_score(y_true_class_flat, y_pred_class_flat),
    }

    print("\nClassification Metrics:")
    for metric, value in classification_metrics.items():
        print(f"{metric}: {value:.16f}")

    torch.save({
        'y_true_reg': y_true_reg_flat,
        'y_pred_reg': y_pred_reg_flat,
        'y_true_class': y_true_class_flat,
        'y_pred_class': y_pred_class_flat,
    }, 'ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')

    print(f"\nTransitions (Input to True Precipitation IN REGRESSION):")
    print(f"Zero to Non-Zero: {input_to_true['zero_to_non_zero']}")
    print(f"Non-Zero to Zero: {input_to_true['non_zero_to_zero']}")
    
    print(f"\nTransitions (Input to Predicted Precipitation):")
    print(f"Zero to Non-Zero: {input_to_pred_REG['zero_to_non_zero']}")
    print(f"Non-Zero to Zero: {input_to_pred_REG['non_zero_to_zero']}")

    print(f"\nTransitions (Input to Predicted Precipitation):")
    print(f"Zero to Non-Zero: {input_to_pred_CLASS['zero_to_non_zero']}")
    print(f"Non-Zero to Zero: {input_to_pred_CLASS['non_zero_to_zero']}")

    return test_total_loss, regression_metrics, classification_metrics


test_total_loss, regression_metrics, classification_metrics = evaluate(
    model=model,
    test_loader=normalized_test_data,
    reg_loss_fn=loss_fn,
    class_loss_fn=bce_loss_fn,
    device=device,
    variable_names=variable_names,
    height=height,
    width=width,
    scaling_factor=scaling_factor,
)

In [ ]:
results = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')

# Access the saved data
y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']
y_true_class = results['y_true_class']
y_pred_class = results['y_pred_class']
regression_metrics = {
    "MSE": mse(y_true_reg, y_pred_reg),
    "MAE": mae(y_true_reg, y_pred_reg),
    "NSE": nash_sutcliffe_efficiency(y_true_reg, y_pred_reg),
    "R2": r2_score(y_true_reg, y_pred_reg),
    "Pearson": pearson_correlation(y_true_reg, y_pred_reg),
    "Spearman": spearman_correlation(y_true_reg, y_pred_reg),
    "NSE": nash_sutcliffe_efficiency(y_true_reg, y_pred_reg),
    "Percentage Error": percentage_error(y_true_reg, y_pred_reg),
    "Percentage Bias": percentage_bias(y_true_reg, y_pred_reg),
    "EMD": earth_movers_distance(y_true_reg, y_pred_reg),
    "Kendall Tau": kendall_tau(y_true_reg, y_pred_reg),
    "Spatial Correlation": spatial_correlation(y_true_reg, y_pred_reg)}

print("\nRegression Metrics:")
for metric, value in regression_metrics.items():
    print(f"{metric}: {value:.16f}")


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Assuming y_pred_original and y_pred_vegetation are your model outputs
def analyze_prediction_changes(y_true, y_pred_original, y_pred_vegetation):
    # Compute differences
    delta_predictions = np.abs(y_pred_original - y_pred_vegetation)
    
    # Summary statistics
    print(f"Mean change: {np.mean(delta_predictions):.4f}")
    print(f"Standard deviation of change: {np.std(delta_predictions):.4f}")
    print(f"Max change: {np.max(delta_predictions):.4f}")
    print(f"Min change: {np.min(delta_predictions):.4f}")

    # Plot histogram of differences
    plt.figure(figsize=(8, 6))
    plt.hist(delta_predictions, bins=50, color='skyblue', edgecolor='black')
    plt.title('Distribution of Prediction Differences')
    plt.xlabel('Absolute Difference in Predictions')
    plt.ylabel('Frequency')
    plt.show()

    # Compute performance metrics
    mae_original = np.mean(np.abs(y_true - y_pred_original))
    mae_vegetation = np.mean(np.abs(y_true - y_pred_vegetation))
    mse_original = np.mean((y_true - y_pred_original)**2)
    mse_vegetation = np.mean((y_true - y_pred_vegetation)**2)

    print(f"\nMAE (Original): {mae_original:.4f}")
    print(f"MAE (With Vegetation): {mae_vegetation:.4f}")
    print(f"MSE (Original): {mse_original:.4f}")
    print(f"MSE (With Vegetation): {mse_vegetation:.4f}")

    # Paired t-test to check statistical significance of prediction changes
    t_stat, p_value = stats.ttest_rel(y_pred_original, y_pred_vegetation)
    print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
    if p_value < 0.05:
        print("Significant difference between predictions with and without vegetation variables (p < 0.05)")
    else:
        print("No significant difference (p >= 0.05)")

results = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')

y_true_reg = results['y_true_reg']
y_pred_original = results['y_pred_reg']

results = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')

y_pred_veg = results['y_pred_reg']

analyze_prediction_changes(y_true_reg, y_pred_original, y_pred_reg)


In [ ]:
import numpy as np
import torch
import scipy.stats as stats
import matplotlib.pyplot as plt

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy()  
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()

delta_y = np.abs(y_pred_original - y_pred_veg)

print(f"Mean change in predictions: {np.mean(delta_y):.4f}")
print(f"Standard deviation of change: {np.std(delta_y):.4f}")
print(f"Max change in predictions: {np.max(delta_y):.4f}")
print(f"Min change in predictions: {np.min(delta_y):.4f}")

plt.figure(figsize=(8, 6))
plt.hist(delta_y, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Prediction Differences')
plt.xlabel('Absolute Difference in Predictions')
plt.ylabel('Frequency')
plt.show()

mae_original = np.mean(np.abs(y_true_reg - y_pred_original))
mae_veg = np.mean(np.abs(y_true_reg - y_pred_veg))
mse_original = np.mean((y_true_reg - y_pred_original) ** 2)
mse_veg = np.mean((y_true_reg - y_pred_veg) ** 2)

print(f"\nMAE (Original): {mae_original:.4f}")
print(f"MAE (With Vegetation): {mae_veg:.4f}")
print(f"MSE (Original): {mse_original:.4f}")
print(f"MSE (With Vegetation): {mse_veg:.4f}")
*
bias_original = np.mean(y_pred_original - y_true_reg)
bias_veg = np.mean(y_pred_veg - y_true_reg)
print(f"\nBias (Original): {bias_original:.4f}")
print(f"Bias (With Vegetation): {bias_veg:.4f}")

pearson_original = stats.pearsonr(y_true_reg.flatten(), y_pred_original.flatten())[0]
pearson_veg = stats.pearsonr(y_true_reg.flatten(), y_pred.flatten())[0]
spearman_original = stats.spearmanr(y_true_reg.flatten(), y_pred_original.flatten())[0]
spearman_veg = stats.spearmanr(y_true_reg.flatten(), y_pred.flatten())[0]

print(f"\nPearson Correlation (Original): {pearson_original:.4f}")
print(f"Pearson Correlation (With Vegetation): {pearson_veg:.4f}")
print(f"Spearman Correlation (Original): {spearman_original:.4f}")
print(f"Spearman Correlation (With Vegetation): {spearman_veg:.4f}")

t_stat, p_value = stats.ttest_rel(y_pred_original.flatten(), y_pred.flatten())
print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("Significant difference between predictions with and without vegetation variables (p < 0.05)")
else:
    print("No significant difference (p >= 0.05)")

wilcoxon_stat, wilcoxon_p = stats.wilcoxon(y_pred_original.flatten(), y_pred.flatten(), zero_method='wilcox', mode='approx')
print(f"\nWilcoxon Test: statistic = {wilcoxon_stat:.4f}, p-value = {wilcoxon_p:.4f}")
if wilcoxon_p < 0.05:
    print("Significant difference (Wilcoxon test, p < 0.05)")
else:
    print("No significant difference (Wilcoxon test, p >= 0.05)")


In [ ]:
#Do the spatial plots 

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy()  # Convert to numpy array
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()

def plot_spatial_heatmap(y_true, y_pred_original, y_pred_veg, grid_shape, timestamp_idx=0):
    y_true_grid = y_true[timestamp_idx].reshape(grid_shape)
    y_pred_orig_grid = y_pred_original[timestamp_idx].reshape(grid_shape)
    y_pred_veg_grid = y_pred_veg[timestamp_idx].reshape(grid_shape)

    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 3, 1)
    plt.title(f"Ground Truth at Time {timestamp_idx}")
    sns.heatmap(y_true_grid, cmap="coolwarm", cbar=True)
    
    plt.subplot(1, 3, 2)
    plt.title(f"Original Model Prediction at Time {timestamp_idx}")
    sns.heatmap(y_pred_orig_grid, cmap="coolwarm", cbar=True)
    
    plt.subplot(1, 3, 3)
    plt.title(f"With Vegetation Prediction at Time {timestamp_idx}")
    sns.heatmap(y_pred_veg_grid, cmap="coolwarm", cbar=True)
    
    plt.show()

def plot_scatter(y_true, y_pred_original, y_pred_veg):
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true.flatten(), y_pred_original.flatten(), label="Original Model", alpha=0.5, color='blue')
    plt.scatter(y_true.flatten(), y_pred.flatten(), label="With Vegetation", alpha=0.5, color='red')
    plt.plot([0, np.max(y_true)], [0, np.max(y_true)], color='black', linestyle='--', label="Perfect Prediction")
    plt.xlabel("True Precipitation (mm)")
    plt.ylabel("Predicted Precipitation (mm)")
    plt.title("Scatter Plot: True vs Predicted Precipitation")
    plt.legend()
    plt.show()

plot_spatial_heatmap(y_true_reg, y_pred_original, y_pred_veg, 81*97, 0)

plot_scatter(y_true_reg, y_pred_original, y_pred_veg)


In [ ]:
#Error heatmaps to see if vegetation changes the spatial distribution of errors

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy()
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()


def plot_error_heatmap(y_true, y_pred, grid_shape, title="Prediction Error"):
    error = np.abs(y_true - y_pred).reshape(grid_shape)
    plt.figure(figsize=(8, 6))
    sns.heatmap(error, cmap="Reds", cbar=True)
    plt.title(title)
    plt.show()

def plot_error_histogram(y_true, y_pred_original, y_pred_veg):
    error_original = y_true - y_pred_original
    error_veg = y_true - y_pred_veg

    plt.figure(figsize=(8, 6))
    plt.hist(error_original.flatten(), bins=50, alpha=0.5, label="Original Model", color='blue')
    plt.hist(error.flatten(), bins=50, alpha=0.5, label="With Vegetation", color='red')
    plt.title("Histogram of Prediction Errors")
    plt.xlabel("Prediction Error (mm)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

plot_error_heatmap(y_true_reg, y_pred_original, 81*97)

plot_error_histogram(y_true_reg, y_pred_original, y_pred_veg)


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()

def plot_classification_rates(y_true_class, y_pred_class):
    """
    Plot the classification metrics: True Positive Rate, False Positive Rate, etc.
    """
    # Ensure y_true_class and y_pred_class are NumPy arrays
    y_true_class = y_true_class.cpu().numpy() if isinstance(y_true_class, torch.Tensor) else y_true_class
    y_pred_class = y_pred_class.cpu().numpy() if isinstance(y_pred_class, torch.Tensor) else y_pred_class

    y_pred_binary = (y_pred_class > 0.5).astype(int)

    tp = np.sum((y_pred_binary == 1) & (y_true_class == 1))
    fp = np.sum((y_pred_binary == 1) & (y_true_class == 0))
    tn = np.sum((y_pred_binary == 0) & (y_true_class == 0))
    fn = np.sum((y_pred_binary == 0) & (y_true_class == 1))

    total = tp + fp + tn + fn
    rates = {
        "True Positive Rate": tp / total,
        "False Positive Rate": fp / total,
        "True Negative Rate": tn / total,
        "False Negative Rate": fn / total
    }

    plt.figure(figsize=(8, 6))
    sns.barplot(x=list(rates.keys()), y=list(rates.values()), palette="viridis")
    plt.title("Classification Rates")
    plt.ylabel("Rate")
    plt.xlabel("Metric")
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.show()

def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title, height=81, width=97):
    """
    Heatmap for spatial precipitation data.
    Assumes the data is provided as a flat array (1D) of size [time_steps * height * width].
    """
    # Reshape the data to (time_steps, height, width)
    num_time_steps = data.size // (height * width)
    data_reshaped = data.reshape((num_time_steps, height, width))

    # Compute mean precipitation across all time steps
    mean_precip = np.mean(data_reshaped, axis=0)

    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

# Plot regression metrics
# Hexbin plot of predicted vs true values
plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_reg, y_pred_reg, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin Plot of Ground Truth vs Predicted Precipitation")
plt.show()

# Histograms
plot_precipitation_distribution(y_true_reg, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_reg, "Predicted Precipitation")

# Scatter plot
plot_scatter(y_true_reg, y_pred_reg, "Scatter Plot of Ground Truth vs Predicted Precipitation")

# Classification metrics plot
plot_classification_rates(y_true_class, y_pred_class)

# Spatial heatmaps for a few predictions (e.g., first 10 grids)
num_visualizations = 10  # Change this to visualize more grids
for i in range(num_visualizations):
    start_idx = i * (81 * 97)
    end_idx = start_idx + (81 * 97)

    # Extract the specific grid
    y_true_grid = y_true_reg[start_idx:end_idx].reshape(81, 97)
    y_pred_grid = y_pred_reg[start_idx:end_idx].reshape(81, 97)
    y_true_class_grid = y_true_class[start_idx:end_idx].reshape(81, 97)
    y_pred_class_grid = y_pred_class[start_idx:end_idx].reshape(81, 97)

    # Plot heatmaps
    plot_spatial_heatmap(y_true_grid, f"Ground Truth Precipitation (Grid {i+1})")
    plot_spatial_heatmap(y_pred_grid, f"Predicted Precipitation (Grid {i+1})")
    plot_spatial_heatmap(y_true_class_grid, f"Ground Truth Zero Classification (Grid {i+1})")
    plot_spatial_heatmap(y_pred_class_grid, f"Predicted Zero Classification (Grid {i+1})")




In [ ]:
results = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')

# Access the saved data
y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']
y_true_class = results['y_true_class']
y_pred_class = results['y_pred_class']

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_true_class, y_pred_class)
f1_scores = 2 * (precision * recall) / (precision + recall)

# Find the threshold with the maximum F1 score
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal Threshold: {optimal_threshold}")
print(f"Precision: {precision[optimal_idx]}")
print(f"Recall: {recall[optimal_idx]}")
print(f"F1 Score: {f1_scores[optimal_idx]}")

# Plot the Precision-Recall curve
plt.plot(recall, precision, label="Precision-Recall Curve")
plt.scatter(recall[optimal_idx], precision[optimal_idx], color='red', label="Optimal Threshold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_true_class, y_pred_class)

plt.plot(fpr, tpr, label="ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()



In [ ]:

total_values = len(y_true_class_flat)
num_zeros = np.sum(y_true_class_flat == 0) 
num_non_zeros = np.sum(y_true_class_flat == 1)

zero_proportion = num_zeros / total_values
non_zero_proportion = num_non_zeros / total_values

print(f"Total values: {total_values}")
print(f"Number of zeros: {num_zeros} ({zero_proportion:.2%})")
print(f"Number of non-zeros: {num_non_zeros} ({non_zero_proportion:.2%})")


In [ ]:
from sklearn.metrics import confusion_matrix

y_pred_binary = (y_pred_class_flat > 0.5).astype(int)
cm = confusion_matrix(y_true_class_flat, y_pred_binary)
print("Confusion Matrix:")
print(cm)


In [ ]:
def plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index, num_samples, height=81, width=97, batch_size=16):
    """
    Visualize the predictions, residuals, and classification for a given batch and selected samples.

    - y_true_reg: Flattened ground truth tensor for regression.
    - y_pred_reg: Flattened predicted tensor for regression.
    - y_true_class: Flattened ground truth tensor for classification.
    - y_pred_class: Flattened predicted tensor for classification.
    - batch_index: Index of the batch to visualize.
    - num_samples: Number of samples from the batch to visualize.
    - height: Height of the grid.
    - width: Width of the grid.
    - batch_size: Number of samples per batch.
    """

    import matplotlib.pyplot as plt
    import numpy as np

    # Compute total samples per batch (height * width * batch_size)
    samples_per_batch = height * width * batch_size

    # Extract the current batch from flattened arrays
    start_idx = batch_index * samples_per_batch
    end_idx = start_idx + samples_per_batch

    true_batch_reg = y_true_reg[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_reg = y_pred_reg[start_idx:end_idx].reshape(batch_size, height, width)
    true_batch_class = y_true_class[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_class = y_pred_class[start_idx:end_idx].reshape(batch_size, height, width)

    # Compute residuals (True - Predicted) for regression
    residuals = true_batch_reg - pred_batch_reg

    # Plot specified number of samples from the batch
    for i in range(min(num_samples, batch_size)):
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        # Plot true regression values
        im1 = axes[0, 0].imshow(true_batch_reg[i], cmap='viridis')
        axes[0, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Regression)")
        plt.colorbar(im1, ax=axes[0, 0])

        # Plot predicted regression values
        im2 = axes[0, 1].imshow(pred_batch_reg[i], cmap='viridis')
        axes[0, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Regression)")
        plt.colorbar(im2, ax=axes[0, 1])

        # Plot regression residuals
        residuals_np = residuals[i]  # Convert to NumPy array
        im3 = axes[0, 2].imshow(
            residuals_np,
            cmap='RdBu',
            vmin=-np.abs(residuals_np).max(),  # Correct handling of max for NumPy arrays
            vmax=np.abs(residuals_np).max()
        )
        axes[0, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (True - Pred)")
        plt.colorbar(im3, ax=axes[0, 2])

        # Plot true classification values
        im4 = axes[1, 0].imshow(true_batch_class[i], cmap='binary')
        axes[1, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Classification)")
        plt.colorbar(im4, ax=axes[1, 0])

        # Plot predicted classification probabilities
        im5 = axes[1, 1].imshow(pred_batch_class[i], cmap='binary')
        axes[1, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Classification)")
        plt.colorbar(im5, ax=axes[1, 1])

        # Plot classification residuals (difference between true and predicted probabilities)
        class_residuals = true_batch_class[i] - pred_batch_class[i]
        class_residuals_np = class_residuals  # Ensure this is a NumPy array
        im6 = axes[1, 2].imshow(
            class_residuals_np,
            cmap='RdBu',
            vmin=-np.abs(class_residuals_np).max(),
            vmax=np.abs(class_residuals_np).max()
        )
        axes[1, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (Classification)")
        plt.colorbar(im6, ax=axes[1, 2])

        plt.tight_layout()
        plt.show()


plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index=0, num_samples=5)



In [ ]:
def plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index, num_samples, threshold=0.5, height=81, width=97, batch_size=16):
    """
    Visualize the predictions, residuals, and classification for a given batch and selected samples.

    - y_true_reg: Flattened ground truth tensor for regression.
    - y_pred_reg: Flattened predicted tensor for regression.
    - y_true_class: Flattened ground truth tensor for classification.
    - y_pred_class: Flattened predicted tensor for classification.
    - batch_index: Index of the batch to visualize.
    - num_samples: Number of samples from the batch to visualize.
    - threshold: Classification threshold for visualization.
    - height: Height of the grid.
    - width: Width of the grid.
    - batch_size: Number of samples per batch.
    """

    import matplotlib.pyplot as plt
    import numpy as np

    # Compute total samples per batch (height * width * batch_size)
    samples_per_batch = height * width * batch_size

    # Extract the current batch from flattened arrays
    start_idx = batch_index * samples_per_batch
    end_idx = start_idx + samples_per_batch

    true_batch_reg = y_true_reg[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_reg = y_pred_reg[start_idx:end_idx].reshape(batch_size, height, width)
    true_batch_class = y_true_class[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_class = y_pred_class[start_idx:end_idx].reshape(batch_size, height, width)

    # Apply the threshold to predicted classification values
    pred_batch_class_thresholded = (pred_batch_class > threshold).float()

    # Compute residuals (True - Predicted) for regression
    residuals = true_batch_reg - pred_batch_reg

    # Plot specified number of samples from the batch
    for i in range(min(num_samples, batch_size)):
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        # Plot true regression values
        im1 = axes[0, 0].imshow(true_batch_reg[i].cpu().numpy(), cmap='viridis')
        axes[0, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Regression)")
        plt.colorbar(im1, ax=axes[0, 0])

        # Plot predicted regression values
        im2 = axes[0, 1].imshow(pred_batch_reg[i].cpu().numpy(), cmap='viridis')
        axes[0, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Regression)")
        plt.colorbar(im2, ax=axes[0, 1])

        # Plot regression residuals
        residuals_np = residuals[i].cpu().numpy()
        im3 = axes[0, 2].imshow(
            residuals_np,
            cmap='RdBu',
            vmin=-np.abs(residuals_np).max(),
            vmax=np.abs(residuals_np).max()
        )
        axes[0, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (Regression)")
        plt.colorbar(im3, ax=axes[0, 2])

        # Plot true classification values
        im4 = axes[1, 0].imshow(true_batch_class[i].cpu().numpy(), cmap='binary')
        axes[1, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Classification)")
        plt.colorbar(im4, ax=axes[1, 0])

        # Plot predicted classification probabilities (thresholded)
        im5 = axes[1, 1].imshow(pred_batch_class_thresholded[i].cpu().numpy(), cmap='binary')
        axes[1, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Classification, Thresholded)")
        plt.colorbar(im5, ax=axes[1, 1])

        # Plot classification residuals (difference between true and thresholded predicted probabilities)
        class_residuals = true_batch_class[i] - pred_batch_class_thresholded[i]
        class_residuals_np = class_residuals.cpu().numpy()
        im6 = axes[1, 2].imshow(
            class_residuals_np,
            cmap='RdBu',
            vmin=-np.abs(class_residuals_np).max(),
            vmax=np.abs(class_residuals_np).max()
        )
        axes[1, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (Classification, Thresholded)")
        plt.colorbar(im6, ax=axes[1, 2])

        plt.tight_layout()
        plt.show()



plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index=0, num_samples=5, threshold=0.5)

